# Build a predictive maintenance model for a delivery company - Classification

In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score, roc_auc_score

In [31]:
data = pd.read_csv("failure.csv")

In [32]:
data.head()

,date,device,failure,attribute1,attribute2,attribute3,attribute4,attribute5,attribute6,attribute7,attribute8,attribute9
0,2015-01-01,S1F01085,0,215630672,56,0,52,6,407438,0,0,7
1,2015-01-01,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0
2,2015-01-01,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0
3,2015-01-01,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0
4,2015-01-01,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3


In [33]:
data

,date,device,failure,attribute1,attribute2,attribute3,attribute4,attribute5,attribute6,attribute7,attribute8,attribute9
0,2015-01-01,S1F01085,0,215630672,56,0,52,6,407438,0,0,7
1,2015-01-01,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0
2,2015-01-01,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0
3,2015-01-01,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0
4,2015-01-01,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...
124489,2015-11-02,Z1F0MA1S,0,18310224,0,0,0,10,353705,8,8,0
124490,2015-11-02,Z1F0Q8RT,0,172556680,96,107,4,11,332792,0,0,13
124491,2015-11-02,Z1F0QK05,0,19029120,4832,0,0,11,350410,0,0,0
124492,2015-11-02,Z1F0QL3N,0,226953408,0,0,0,12,358980,0,0,0


In [34]:
data.isnull().sum()

date          0
device        0
failure       0
attribute1    0
attribute2    0
attribute3    0
attribute4    0
attribute5    0
attribute6    0
attribute7    0
attribute8    0
attribute9    0
dtype: int64

In [35]:
y = data['failure']

In [36]:
#    - Zaman özellikleri çıkar
data['date'] = pd.to_datetime(data['date'])
data['day_of_week'] = data['date'].dt.dayofweek
data['month'] = data['date'].dt.month

In [37]:
data

,date,device,failure,attribute1,attribute2,attribute3,attribute4,attribute5,attribute6,attribute7,attribute8,attribute9,day_of_week,month
0,2015-01-01,S1F01085,0,215630672,56,0,52,6,407438,0,0,7,3,1
1,2015-01-01,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0,3,1
2,2015-01-01,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0,3,1
3,2015-01-01,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0,3,1
4,2015-01-01,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124489,2015-11-02,Z1F0MA1S,0,18310224,0,0,0,10,353705,8,8,0,0,11
124490,2015-11-02,Z1F0Q8RT,0,172556680,96,107,4,11,332792,0,0,13,0,11
124491,2015-11-02,Z1F0QK05,0,19029120,4832,0,0,11,350410,0,0,0,0,11
124492,2015-11-02,Z1F0QL3N,0,226953408,0,0,0,12,358980,0,0,0,0,11


In [39]:
# Cihaz geçmiş arıza oranı özelliği (summary)
device_stats = data.groupby('device')['failure'].agg(['sum','count']).reset_index().rename(columns={'sum':'fail_sum','count':'fail_count'})
device_stats['fail_rate'] = device_stats['fail_sum'] / device_stats['fail_count']
data = data.merge(device_stats[['device','fail_rate']], on='device', how='left')

In [40]:
data

,date,device,failure,attribute1,attribute2,attribute3,attribute4,attribute5,attribute6,attribute7,attribute8,attribute9,day_of_week,month,fail_rate_x,fail_rate_y
0,2015-01-01,S1F01085,0,215630672,56,0,52,6,407438,0,0,7,3,1,0.0,0.0
1,2015-01-01,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0,3,1,0.0,0.0
2,2015-01-01,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0,3,1,0.0,0.0
3,2015-01-01,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0,3,1,0.0,0.0
4,2015-01-01,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3,3,1,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124489,2015-11-02,Z1F0MA1S,0,18310224,0,0,0,10,353705,8,8,0,0,11,0.0,0.0
124490,2015-11-02,Z1F0Q8RT,0,172556680,96,107,4,11,332792,0,0,13,0,11,0.0,0.0
124491,2015-11-02,Z1F0QK05,0,19029120,4832,0,0,11,350410,0,0,0,0,11,0.0,0.0
124492,2015-11-02,Z1F0QL3N,0,226953408,0,0,0,12,358980,0,0,0,0,11,0.0,0.0


In [41]:
# Girdi sütunları belirle
drop_cols = ['date', 'device', 'failure']
# kategorik kolonlar: 'month', 'day_of_week'
categorical = ['month','day_of_week']
numerical = [c for c in data.select_dtypes(include=[np.number]).columns if c not in ['failure'] + categorical]

In [43]:
preproc = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical)
])

In [44]:
# Model pipeline: preproc → PCA → SMOTE+Tomek → XGBoost
pipeline = ImbPipeline(steps=[
    ('preproc', preproc),
    ('pca', PCA(n_components=0.95, svd_solver='full')),
    ('smote_tomek', SMOTETomek(random_state=42)),
    ('clf', XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        scale_pos_weight = (data['failure']==0).sum() / (data['failure']==1).sum()
    ))
])

In [45]:
# Çapraz doğrulama ile değerlendirme
scoring = {
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score),
    'f1': make_scorer(f1_score),
    'roc_auc': 'roc_auc'
}

In [46]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(pipeline, data.drop(columns=drop_cols), y, cv=cv, scoring=scoring, return_train_score=False)

C:\Users\kosey\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [00:38:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\kosey\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [00:39:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\kosey\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [00:40:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\kosey\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [00:41:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.

In [47]:
print("Recall:", np.mean(scores['test_recall']))
print("Precision:", np.mean(scores['test_precision']))
print("F1:", np.mean(scores['test_f1']))
print("ROC AUC:", np.mean(scores['test_roc_auc']))

Recall: 0.29264069264069265
Precision: 0.0839724915222336
F1: 0.13014765762290514
ROC AUC: 0.9009186261468866
